# RL Post-Training GPU Memory Profile (veRL track)

veRL (HybridFlow) orchestrates rollout and training via Ray + vLLM, which doesn't run cleanly inside a hosted notebook — so this notebook doesn't run the training itself. Instead it analyzes traces produced on a GPU box (local or rented: RunPod/Lambda/Vast.ai) via the repo's scripts.

**Before running this notebook:**

1. Install veRL per its own docs: https://github.com/volcengine/verl
2. Wire the two phase markers (and optionally the category-breakdown calls) into veRL's trainer loop, as described in `scripts/memory_poller.py`'s `run_verl_track` docstring.
3. Run `python scripts/memory_poller.py --track verl --config <your_config>.yaml --steps 20 --out results/verl_trace.csv` on the GPU box.
4. Copy `results/verl_trace.csv` (and `results/verl_breakdown.csv`, if you wired the breakdown calls) into this repo, then run the cells below.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

TRACE_PATH = "../results/verl_trace.csv"
BREAKDOWN_PATH = "../results/verl_breakdown.csv"

assert os.path.exists(TRACE_PATH), (
    f"{TRACE_PATH} not found — run scripts/memory_poller.py --track verl on a GPU box first "
    "and copy the resulting CSV here (see markdown above)."
)

df = pd.read_csv(TRACE_PATH)
df["t"] = df["timestamp"] - df["timestamp"].min()
df["allocated_gb"] = df["allocated_bytes"] / 1e9
df.tail()

## Baseline phase memory profile (Experiment 1)

In [ ]:
PHASE_COLORS = {"rollout": "#4C72B0", "update": "#DD8452", "idle": "#B0B0B0"}

fig, ax = plt.subplots(figsize=(10, 4))
for phase, group in df.groupby("phase"):
    ax.scatter(group["t"], group["allocated_gb"], s=4, label=phase, color=PHASE_COLORS.get(phase, "#333"))
ax.set_xlabel("time (s)")
ax.set_ylabel("GPU memory allocated (GB)")
ax.set_title("veRL — memory over rollout/update phases")
ax.legend()
fig.tight_layout()
plt.show()

## Idle-time characterization (Experiment 3)

Same simplifying proxy as the TRL notebook: each phase's wall-clock share approximates the idle time of the category that's untouched during it.

In [ ]:
sample_interval = df["t"].diff().median()  # seconds between polls, inferred from the trace
phase_time = df.groupby("phase")["t"].count() * sample_interval
total_time = phase_time.sum()

rollout_time = phase_time.get("rollout", 0.0)
update_time = phase_time.get("update", 0.0)

print(f"rollout phase: {rollout_time:.2f}s ({rollout_time / total_time:.1%} of total)")
print(f"update phase:  {update_time:.2f}s ({update_time / total_time:.1%} of total)")

## Memory category breakdown (Experiment 2, optional)

Only runs if `results/verl_breakdown.csv` was produced (see setup instructions above).

In [ ]:
if os.path.exists(BREAKDOWN_PATH):
    bdf = pd.read_csv(BREAKDOWN_PATH)
    for col in ["parameters", "gradients", "optimizer_state"]:
        bdf[col] = bdf[col] / 1e9
    avg_by_checkpoint = bdf.groupby("checkpoint")[["parameters", "gradients", "optimizer_state"]].mean()
    display(avg_by_checkpoint)
    ax = avg_by_checkpoint.plot(kind="bar", figsize=(7, 4))
    ax.set_ylabel("GB")
    ax.set_title("veRL — category breakdown by checkpoint")
    plt.tight_layout()
    plt.show()
else:
    print(f"{BREAKDOWN_PATH} not found — skipping category breakdown. See setup instructions above.")

## References

See `notebooks/01_trl_baseline_profile.ipynb` or the repo [README](https://github.com/rahulsanjay009/rl-memory-profiling) for the full cited related work.